# 01A — Microsoft Optical sector pack

**Outcome:** translate a reproducible subset of Microsoft's public optical-backbone telemetry into the common Pack interface.

This is a **real-data realism test**, not a labelled anomaly benchmark. Microsoft removed outage days from the public release, so this notebook creates `PACK-CORE`, topology and time splits, but deliberately creates no `PACK-EVAL`. Raw files and derived telemetry must not be committed to Git.

The default fixture keeps the full history of 48 real channels: 12 channels from each of four deterministically selected optical segments. Increase the two sampling controls only after the first end-to-end run succeeds.

## 1. Setup

In [ ]:
import hashlib
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

default_data_root = (Path("/content/drive/MyDrive/anomaly_detection")
                     if IN_COLAB else Path.home() / "anomaly_detection_data")
DATA_ROOT = Path(os.getenv("ANOMALY_DATA_ROOT")
                 or os.getenv("ANOMALY_DRIVE_ROOT")
                 or default_data_root).expanduser()
default_code_root = (DATA_ROOT / "research" / "milestone1" if IN_COLAB
                     else Path.cwd() if (Path.cwd() / "milestone1_core.py").is_file()
                     else Path.cwd() / "notebooks" / "drive_research")
NOTEBOOK_HOME = Path(os.getenv("ANOMALY_NOTEBOOK_HOME", default_code_root)).expanduser()
if not (NOTEBOOK_HOME / "milestone1_core.py").is_file():
    raise FileNotFoundError(f"milestone1_core.py was not found in {NOTEBOOK_HOME}")
if str(NOTEBOOK_HOME) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_HOME))

from milestone1_core import (
    OPTIONAL_CORE_SCHEMAS, PACK_SCHEMAS, SPLIT_SCHEMAS,
    read_pack, save_pack, source_file,
)

SOURCE = Path(os.getenv(
    "MICROSOFT_OPTICAL_SOURCE_ROOT",
    DATA_ROOT / "sources" / "microsoft_optical" / "raw" / "plots_dataset_release",
)).expanduser()
PACK_RUN_ID = os.getenv("MICROSOFT_OPTICAL_PACK_RUN_ID", "microsoft_optical_pack_v0_1_0")
PACK_ROOT = DATA_ROOT / "outputs" / "packs" / "microsoft_optical" / PACK_RUN_ID
SEGMENT_LIMIT = int(os.getenv("MICROSOFT_OPTICAL_SEGMENTS", "4"))
CHANNELS_PER_SEGMENT = int(os.getenv("MICROSOFT_OPTICAL_CHANNELS_PER_SEGMENT", "12"))
RUN_BUILD = os.getenv("RUN_MICROSOFT_OPTICAL_PACK", "1") == "1"
LICENSE_REVIEWED = os.getenv("MICROSOFT_OPTICAL_LICENSE_REVIEWED", "0") == "1"

display(pd.Series({
    "source": str(SOURCE),
    "pack_root": str(PACK_ROOT),
    "segments_requested": SEGMENT_LIMIT,
    "channels_per_segment": CHANNELS_PER_SEGMENT,
    "license_reviewed": LICENSE_REVIEWED,
}, name="value").to_frame())

if not LICENSE_REVIEWED:
    print("LICENCE NOTICE — local research use only until you review license.docx.")
    print("Do not redistribute the raw data or derived row-level telemetry.")

## 2. Source vocabulary

The release contains one text file per optical channel. The filename supplies the channel and segment identities; each row contains a timestamp and four measurements. `?` means an attempted measurement with no numeric value. No undocumented physical bounds or anomaly labels are introduced.

In [ ]:
NATIVE_COLUMNS = [
    "timestamp", "q_factor", "transmit_power_dbm",
    "chromatic_dispersion_ps_nm", "polarization_mode_dispersion_ps",
]
METRIC_COLUMNS = ["native_field", *PACK_SCHEMAS["metric_catalogue"]]
CADENCE_SECONDS = 15 * 60
metric_map = pd.DataFrame([
    ("q_factor", "q_factor", "optical_channel", "gauge", "ratio", "periodic", CADENCE_SECONDS),
    ("transmit_power_dbm", "transmit_power_dbm", "optical_channel", "gauge", "dBm", "periodic", CADENCE_SECONDS),
    ("chromatic_dispersion_ps_nm", "chromatic_dispersion_ps_nm", "optical_channel", "gauge", "ps/nm", "periodic", CADENCE_SECONDS),
    ("polarization_mode_dispersion_ps", "polarization_mode_dispersion_ps", "optical_channel", "gauge", "ps", "periodic", CADENCE_SECONDS),
], columns=METRIC_COLUMNS)
assert metric_map["metric_id"].is_unique
display(metric_map)

## 3. Inventory and freeze the population

Selection is stable: segment and channel identifiers are ordered by SHA-256, not by the result. Raising either limit produces a reproducible larger experiment. The current fixture remains explicitly a subset of the 4,000-channel release.

In [ ]:
FILE_PATTERN = re.compile(r"channel_(\d+)_segment_(\d+)\.txt")
EXPECTED_FILES = 4_000

def stable_key(value):
    return hashlib.sha256(str(value).encode()).hexdigest()

def release_directory(path):
    path = Path(path)
    if any(path.glob("channel_*_segment_*.txt")):
        return path
    matches = [folder for folder in path.rglob("plots_dataset_release") if folder.is_dir()]
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected one plots_dataset_release folder under {path}")
    return matches[0]

SOURCE = release_directory(SOURCE)
records = []
for path in SOURCE.glob("channel_*_segment_*.txt"):
    match = FILE_PATTERN.fullmatch(path.name)
    if match:
        channel_id, segment_id = map(int, match.groups())
        records.append({"path": path, "channel_id": channel_id, "segment_id": segment_id})
inventory = pd.DataFrame(records)
if len(inventory) != EXPECTED_FILES:
    raise ValueError(f"Expected {EXPECTED_FILES:,} channel files; found {len(inventory):,}")
if inventory["channel_id"].duplicated().any():
    raise ValueError("Channel identifiers are not unique in the source release")

segment_sizes = inventory.groupby("segment_id").size().rename("available_channels")
eligible_segments = segment_sizes.loc[segment_sizes.ge(CHANNELS_PER_SEGMENT)].index.tolist()
selected_segments = sorted(eligible_segments, key=stable_key)[:SEGMENT_LIMIT]
if len(selected_segments) < SEGMENT_LIMIT:
    raise ValueError("Too few segments contain the requested number of channels")
selected = (inventory.loc[inventory["segment_id"].isin(selected_segments)]
            .assign(selection_key=lambda frame: frame["channel_id"].map(stable_key))
            .sort_values(["segment_id", "selection_key"])
            .groupby("segment_id", as_index=False).head(CHANNELS_PER_SEGMENT)
            .sort_values(["segment_id", "channel_id"]).reset_index(drop=True))
selected["entity_id"] = selected["channel_id"].map(lambda value: f"optical-channel-{value:04d}")
selected["episode_id"] = selected["entity_id"] + "::released-history"
selected["group_id"] = selected["segment_id"].map(lambda value: f"optical-segment-{value:03d}")

display(segment_sizes.describe().to_frame().T)
display(selected[["entity_id", "group_id", "path"]].head(20))
print(f"Selected {len(selected)} channels from {len(selected_segments)} segments")

## 4. Translate observations

Repeated source timestamps are resolved per metric. Identical repeats are coalesced. A repeated timestamp containing disagreement or a `?` is retained once as `invalid`; its value is not guessed. A metric that never reports a numeric value for a channel is omitted for that episode, which means *not available*, not *invalid forever*.

In [ ]:
translation_audit = {
    "source_rows": 0,
    "duplicate_source_rows_coalesced": 0,
    "invalid_metric_observations": 0,
    "unavailable_episode_metric_pairs": 0,
}

def read_channel(path):
    frame = pd.read_csv(
        path, sep=r"\s+", header=None, names=NATIVE_COLUMNS,
        na_values="?", keep_default_na=True,
    )
    frame["timestamp"] = pd.to_datetime(
        frame["timestamp"], format="%Y.%m.%d.%H.%M.%S", errors="raise",
    ).dt.tz_localize("UTC")
    for column in NATIVE_COLUMNS[1:]:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame

def resolve_metric(frame, native_field, metric_id, entity_id, episode_id):
    if not frame[native_field].notna().any():
        translation_audit["unavailable_episode_metric_pairs"] += 1
        return None
    grouped = frame.groupby("timestamp", sort=True)[native_field]
    resolved = grouped.agg(value="first", attempts="size", valid="count", distinct="nunique").reset_index()
    invalid = resolved["valid"].ne(resolved["attempts"]) | resolved["distinct"].ne(1)
    resolved.loc[invalid, "value"] = np.nan
    translation_audit["invalid_metric_observations"] += int(invalid.sum())
    return pd.DataFrame({
        "event_ts": resolved["timestamp"],
        "entity_id": entity_id,
        "episode_id": episode_id,
        "metric_id": metric_id,
        "value": resolved["value"],
        "quality_code": np.where(invalid, "invalid", "measured"),
    })

def telemetry_batches(selection):
    for row in selection.itertuples(index=False):
        frame = read_channel(row.path)
        translation_audit["source_rows"] += len(frame)
        translation_audit["duplicate_source_rows_coalesced"] += len(frame) - frame["timestamp"].nunique()
        parts = []
        for metric in metric_map.itertuples(index=False):
            part = resolve_metric(
                frame, metric.native_field, metric.metric_id,
                row.entity_id, row.episode_id,
            )
            if part is not None:
                parts.append(part)
        if parts:
            yield pd.concat(parts, ignore_index=True)

def source_time_span(selection):
    starts, ends = [], []
    for path in selection["path"]:
        values = pd.read_csv(path, sep=r"\s+", header=None, usecols=[0]).iloc[:, 0]
        timestamps = pd.to_datetime(values, format="%Y.%m.%d.%H.%M.%S", errors="raise", utc=True)
        starts.append(timestamps.min())
        ends.append(timestamps.max())
    return min(starts), max(ends)

def time_partitions(start, end):
    final_end = end + pd.Timedelta(seconds=CADENCE_SECONDS)
    span = (final_end - start).total_seconds()
    development_start = start + pd.Timedelta(seconds=round(span * 0.50 / CADENCE_SECONDS) * CADENCE_SECONDS)
    holdout_start = start + pd.Timedelta(seconds=round(span * 0.75 / CADENCE_SECONDS) * CADENCE_SECONDS)
    return pd.DataFrame([
        ("calibration", start, development_start, "microsoft_optical_time_v1"),
        ("development", development_start, holdout_start, "microsoft_optical_time_v1"),
        ("holdout", holdout_start, final_end, "microsoft_optical_time_v1"),
    ], columns=SPLIT_SCHEMAS["time_partitions"])

source_start, source_end = source_time_span(selected)
splits = {"time_partitions": time_partitions(source_start, source_end)}
display(splits["time_partitions"])

## 5. Build the Pack

In [ ]:
entities = selected[["entity_id"]].assign(entity_type="optical_channel")
entities = entities[PACK_SCHEMAS["entity_registry"]].drop_duplicates()
episodes = selected[["episode_id", "entity_id"]].assign(episode_basis="released_channel_history")
episodes = episodes[PACK_SCHEMAS["observation_episodes"]].drop_duplicates()
topology = selected[["entity_id", "group_id"]].assign(
    group_type="optical_segment", hierarchy_level=0, group_family="physical_topology",
)[OPTIONAL_CORE_SCHEMAS["topology_memberships"]].drop_duplicates()
catalogue = metric_map[PACK_SCHEMAS["metric_catalogue"]].copy()

selected_source_files = [source_file(path, SOURCE) for path in selected["path"]]
license_path = SOURCE / "license.docx"
if not license_path.is_file():
    raise FileNotFoundError(f"The release licence is missing: {license_path}")
selected_source_files.append(source_file(license_path, SOURCE, role="license"))
source_info = {
    "source_id": "microsoft_optical_data_v1",
    "publisher": "Microsoft",
    "download_page": "https://www.microsoft.com/en-us/download/details.aspx?id=54267",
    "official_archive_sha256": "ebd080a040c9eadd6cced37ef3cf2e6f25c0560d6f1034dbc0e9f22541bf2f1d",
    "release_file_count": len(inventory),
    "selected_channels": len(selected),
    "selected_segments": len(selected_segments),
    "selection_rule": "sha256_order_segments_then_channels_v1",
    "timestamp_semantics": "source_timezone_undisclosed; UTC used only as a canonical ordering coordinate",
    "license_reviewed_by_runner": LICENSE_REVIEWED,
    "translation_audit": translation_audit,
    "files": selected_source_files,
}

if RUN_BUILD:
    pack_manifest = save_pack(
        PACK_ROOT,
        sector="microsoft_optical",
        pack_version="0.1.0",
        source_info=source_info,
        telemetry=telemetry_batches(selected),
        catalogue=catalogue,
        entities=entities,
        episodes=episodes,
        topology=topology,
        splits=splits,
        evaluation=None,
        notes=[
            "Real production telemetry; no public incident truth.",
            "Outage days were removed by the publisher for confidentiality.",
            "No physical bounds or anomaly labels were inferred.",
        ],
    )
else:
    pack_manifest = read_pack(PACK_ROOT)

assert not pack_manifest["evaluation_tables"]
assert pack_manifest["capabilities"]["topology"]
display(pd.Series(pack_manifest, name="value").to_frame())
display(pd.Series(pack_manifest["source"]["translation_audit"], name="count").to_frame())
print("Pack root:", PACK_ROOT)
print("Next: run 01B with SECTOR = 'microsoft_optical'")